### RAG PIPELINE - DATA INGESTION to VECTOR DATABASE PIPELINE


In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\rakes\AppData\Local\Temp\ipykernel_21016\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [2]:
###READ PDF FILES

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    if not pdf_dir.is_dir():
        raise NotADirectoryError(f"PDF directory does not exist: {pdf_dir.resolve()}")
    
    # Find all PDF files recursively
    pdf_files = sorted(
        path for path in pdf_dir.rglob("*") if path.is_file() and path.suffix.lower() == ".pdf"
    )
    
    print(f"Found {len(pdf_files)} PDF files to process in {pdf_dir.resolve()}")
    failures = []
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
                
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
            failures.append((pdf_file, e))
            
    if failures:
        failed_files = ", ".join(str(path) for path, _ in failures)
        raise RuntimeError(f"Failed to process {len(failures)} PDF file(s): {failed_files}") from failures[0][1]
            
    return all_documents

def find_pdf_directory():
    """Find this project's PDF folder from the current or a parent directory."""
    working_directory = Path.cwd().resolve()
    for project_root in (working_directory, *working_directory.parents):
        for candidate in (project_root / "notebook" / "data" / "pdf", project_root / "data" / "pdf"):
            if candidate.is_dir():
                return candidate
    raise FileNotFoundError(
        f"Could not find 'notebook/data/pdf' or 'data/pdf' from {working_directory} or its parents"
    )

all_processed_documents = process_all_pdfs(find_pdf_directory())

Found 3 PDF files to process in C:\Users\rakes\OneDrive\Desktop\projects\rag\notebook\data\pdf

Processing: LangChain_Complete_Notes_Guide.pdf
Loaded 8 pages

Processing: LLM_Complete_Notes_Guide.pdf
Loaded 6 pages

Processing: sample.pdf
Loaded 12 pages


In [3]:
all_processed_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-03T07:21:35+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-11-03T07:21:35+00:00', 'subject': '(unspecified)', 'title': 'LANGCHAIN', 'trapped': '/False', 'source': 'C:\\Users\\rakes\\OneDrive\\Desktop\\projects\\rag\\notebook\\data\\pdf\\LangChain_Complete_Notes_Guide.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'LangChain_Complete_Notes_Guide.pdf', 'file_type': 'pdf'}, page_content='LANGCHAIN\nPage 1\n LANGCHAIN\n Complete developer notes: models, prompts, runnables, RAG, tools, agents, LangGraph, and\n deployment\nHow to use these notes\nRead the concepts in order, run the code snippets in a small project, and verify every claim with experiments. API names can\nchange across library versions; consult the installed package documentation when implementing.'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource

In [4]:
### TEXT SPLITTING getting the text from the pdf files and splitting it into chunks
def split_into_chunks(all_processed_documents):
    """Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    
    all_chunks = []
    for doc in all_processed_documents:
        chunks = text_splitter.split_documents([doc])
        all_chunks.extend(chunks)
    
    print(f"Total chunks created: {len(all_chunks)}")
    return all_chunks    

all_chunks = split_into_chunks(all_processed_documents)

all_chunks

Total chunks created: 79


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-11-03T07:21:35+00:00', 'author': 'OpenAI', 'keywords': '', 'moddate': '2026-11-03T07:21:35+00:00', 'subject': '(unspecified)', 'title': 'LANGCHAIN', 'trapped': '/False', 'source': 'C:\\Users\\rakes\\OneDrive\\Desktop\\projects\\rag\\notebook\\data\\pdf\\LangChain_Complete_Notes_Guide.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'LangChain_Complete_Notes_Guide.pdf', 'file_type': 'pdf'}, page_content='LANGCHAIN\nPage 1\n LANGCHAIN\n Complete developer notes: models, prompts, runnables, RAG, tools, agents, LangGraph, and\n deployment\nHow to use these notes\nRead the concepts in order, run the code snippets in a small project, and verify every claim with experiments. API names can\nchange across library versions; consult the installed package documentation when implementing.'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource

###EMBEDDING AND VECTOR STORE BD


In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\rakes\OneDrive\Desktop\projects\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initializes the embedding manager.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """
        Loads the sentence transformer model.
        """
        print(f"Loading embedding model: {self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print("Model loaded successfully!")
        
        # Get and print embedding dimensions
        dimension = self.model.get_sentence_embedding_dimension()
        print(f"Embedding dimension: {dimension}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generates embeddings for a list of texts.
        """
        embeddings = self.model.encode(texts, show_progress_bar=True)
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager

    

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2944.39it/s]


Model loaded successfully!
Embedding dimension: 384


C:\Users\rakes\AppData\Local\Temp\ipykernel_21016\1804245460.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.model.get_sentence_embedding_dimension()


###VECTOR STORE

In [7]:
import os

class VectorStoreManager:
    def __init__(self, collection_name: str = "pdf_documents", persistent_directory: str = "./data/vector_store"):
        """
        Initializes the vector store manager and persistent ChromaDB client.
        """
        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """
        Creates the persistent directory, Chroma client, and collection.
        """
        os.makedirs(self.persistent_directory, exist_ok=True)
        
        # Initialize persistent ChromaDB client
        self.client = chromadb.PersistentClient(path=self.persistent_directory)
        
        # Get or create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "RAG vector store collection"}
        )
        
        print(f"Vector store initialized. Existing documents in collection: {self.collection.count()}")

    def add_documents(self, documents: List[Any], embeddings: List[List[float]]):
        """
        Adds document chunks, their corresponding embeddings, and metadata to the vector store.
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match the number of embedding vectors.")

        ids = []
        embedding_list = []
        metadata_list = []
        document_texts = []

        for idx, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID for each document record
            doc_id = str(uuid.uuid4())
            ids.append(doc_id)
            
            # Convert numpy/tensor embedding to a standard list
            embedding_list.append(embedding.tolist() if hasattr(embedding, "tolist") else embedding)
            
            # Extract and enrich metadata
            metadata = doc.metadata.copy() if hasattr(doc, "metadata") and doc.metadata else {}
            metadata["doc_index"] = idx
            metadata["content_length"] = len(doc.page_content)
            metadata_list.append(metadata)
            
            # Store raw text content
            document_texts.append(doc.page_content)

        try:
            # Add records to the ChromaDB collection all at once outside the loop
            self.collection.add(
                ids=ids,
                embeddings=embedding_list,
                metadatas=metadata_list,
                documents=document_texts
            )
            print(f"Successfully added {len(documents)} documents. Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")

        return ids

# Instantiate the manager
vector_store_manager = VectorStoreManager()
vector_store_manager


Vector store initialized. Existing documents in collection: 0


In [8]:
### Convert text chunks into embeddings and store them in ChromaDB

# Recreate chunks from the loaded PDF pages.
if "all_processed_documents" not in globals():
    all_processed_documents = process_all_pdfs(find_pdf_directory())

all_chunks = split_into_chunks(all_processed_documents)

if not all_chunks:
    raise ValueError("No text chunks were created. Check whether the PDFs contain extractable text.")

# Extract text from each LangChain Document.
texts = [chunk.page_content.strip() for chunk in all_chunks]
valid_items = [(chunk, text) for chunk, text in zip(all_chunks, texts) if text]

if not valid_items:
    raise ValueError("All extracted chunks are empty. The PDFs may be scanned images; run OCR first.")

all_chunks = [chunk for chunk, _ in valid_items]
texts = [text for _, text in valid_items]

# Generate one embedding for every chunk.
embeddings = embedding_manager.generate_embeddings(texts)

if len(embeddings) != len(all_chunks):
    raise ValueError(
        f"Embedding count ({len(embeddings)}) does not match chunk count ({len(all_chunks)})."
    )

# Store chunks and embeddings in ChromaDB.
# Chroma metadata values must be strings, numbers, booleans, or non-empty lists
# of those types; remove None values and normalize unsupported values.
ids = []
documents = []
metadatas = []
embedding_list = []

for index, (chunk, embedding) in enumerate(zip(all_chunks, embeddings)):
    metadata = {}

    for key, value in (chunk.metadata or {}).items():
        if value is None:
            continue
        if isinstance(value, (str, int, float, bool)):
            metadata[key] = value
        else:
            metadata[key] = str(value)

    metadata["chunk_index"] = index
    metadata["content_length"] = len(chunk.page_content)

    ids.append(str(uuid.uuid4()))
    documents.append(chunk.page_content)
    metadatas.append(metadata)
    embedding_list.append(embedding.tolist() if hasattr(embedding, "tolist") else list(embedding))

# Use batches to avoid oversized ChromaDB requests on larger PDF collections.
batch_size = 100
for start in range(0, len(ids), batch_size):
    end = start + batch_size
    vector_store_manager.collection.add(
        ids=ids[start:end],
        embeddings=embedding_list[start:end],
        metadatas=metadatas[start:end],
        documents=documents[start:end],
    )

print(f"Chunks prepared: {len(all_chunks)}")
print(f"Embeddings generated: {len(embedding_list)}")
print(f"Records stored in ChromaDB: {vector_store_manager.collection.count()}")


Total chunks created: 79


Batches: 100%|██████████| 3/3 [00:03<00:00,  1.08s/it]

Chunks prepared: 79
Embeddings generated: 79
Records stored in ChromaDB: 79


In [9]:
texts

['LANGCHAIN\nPage 1\n LANGCHAIN\n Complete developer notes: models, prompts, runnables, RAG, tools, agents, LangGraph, and\n deployment\nHow to use these notes\nRead the concepts in order, run the code snippets in a small project, and verify every claim with experiments. API names can\nchange across library versions; consult the installed package documentation when implementing.',
 'LANGCHAIN\nPage 2\n1. What is LangChain?\nLangChain is an open-source framework for building applications powered by language models. It provides integrations and\nabstractions for models, prompts, documents, retrievers, tools, agents, and application workflows. It helps connect\ncomponents, but it does not replace understanding the underlying LLM and RAG concepts.\nLangChain is an orchestration framework—not an LLM, vector database, or guarantee of answer quality. Its abstractions are\nuseful when they reduce integration work; direct SDK calls may be simpler for small applications.\n2. Understand the ecosy